In [0]:
%run ../read_params

In [0]:
%run ../utils

In [0]:
%run ./schemas

In [0]:
def species_extraction(url, schema, base_dict):
    """
    Doc String
    """
    species_df = api_extraction(url, schema, base_dict)

    # species_df.write.format('delta').mode("append").option('mergeSchema', 'true').saveAsTable(f"{STAGING_DATABASE_PREFIX}.species")

    return species_df

In [0]:
def varieties_extraction(species_df, base_dict):
    """
    Doc String
    """
    varieties = species_df.select('varieties').collect()[0][0]

    varieties_list = [[row['is_default'], row['pokemon']['name'], row['pokemon']['url']] for row in varieties]

    varieties_df_list = []

    for is_default, variety_name, varieties_url in varieties_list:

        varieties_df_sub = api_extraction(varieties_url, varieties_schema, base_dict)

        varieties_df_list.append(varieties_df_sub)

    varieties_df = reduce(DataFrame.union, varieties_df_list)

    varieties_df.write.format('delta').mode("append").option('mergeSchema', 'true').saveAsTable(f"{STAGING_DATABASE_PREFIX}.varieties")

    return varieties_df

In [0]:
def form_data_extraction(varieties_df, base_dict):
    """
    Doc String
    """
    form_varieties = (varieties_df
                      .withColumn('form', explode('forms'))
                      .select(
                        'id', 
                        col('form.url').alias('form_url')
                      )
    )

    form_urls = [[row['id'], row['form_url']] for row in form_varieties.collect()]

    forms_df_list = []

    for variety_id, form_url in form_urls:
        base_dict.update({'variety_id' : variety_id})

        form_df_sub = api_extraction(form_url, form_schema, base_dict)

        forms_df_list.append(form_df_sub)

    form_df = reduce(DataFrame.union, forms_df_list)

    form_df.write.format('delta').mode("append").option('mergeSchema', 'true').saveAsTable(f"{STAGING_DATABASE_PREFIX}.forms")

    return form_df

In [0]:
def obtain_pokemon_extraction_list(lookup_table_name):
    pokemon_entries = spark.sql(f"""
        SELECT
            pokemon_entries 
        FROM 
            {STAGING_DATABASE_PREFIX}.nat_dex
        WHERE 
            id = 1
    """).collect()[0][0]

    pokemon_entries_list_total = [[row['entry_number'], row['pokemon_species']['name'], row['pokemon_species']['url']] for row in pokemon_entries]

    existing_pokedex_list = check_existence(lookup_table_name, 'pokedex_no')

    pokemon_entries_list = [entry for entry in pokemon_entries_list_total if entry[0] not in existing_pokedex_list]

    total_entries = len(pokemon_entries_list)

    return pokemon_entries_list, total_entries

In [0]:
LOOKUPS_TABLE_NAME = 'pokedex_updated'

pokemon_entries_list, total_entries = obtain_pokemon_extraction_list(LOOKUPS_TABLE_NAME)

num_entries_complete = 0

for pokedex_no, name, species_url in pokemon_entries_list[2:3]:
    print(f"Starting {pokedex_no}: {name}")
    base_dict = {
        'nat_dex_pokedex_no': pokedex_no,
        'nat_dex_pokemon_name': name
    }

    species_df = species_extraction(species_url, species_schema, base_dict)

    base_dict.update({'species_id' : species_df.select('id').collect()[0][0]})

    varieties_df = varieties_extraction(species_df, base_dict)

    form_df = form_data_extraction(varieties_df, base_dict)

    # insert_query = f"""
    #     INSERT INTO {LOOKUPS_DATABASE_PREFIX}.pokedex_updated
    #     VALUES ({pokedex_no}, '{name}', '{datetime.now(timezone.utc).replace(tzinfo = None)}')
    # """
    # spark.sql(insert_query)

    num_entries_complete += 1

    print(f"Completed {pokedex_no}: {name} - {num_entries_complete}/{total_entries}")

In [0]:
species_df.select('varieties').collect()[0][0]


In [0]:
form_varieties = (varieties_df
                    .withColumn('form', explode('forms'))
                    .select(
                    'id', 
                    col('form.url').alias('form_url')
                    )
)

form_varieties.display()

In [0]:
form_df.display()